## Day 3 - Part 3: Seq2Seq와 어텐션: 번역기를 만들어보자

### 개요

Day 3의 Part 2에서 우리는 LSTM을 사용하여 문장의 '감성'을 긍정 또는 부정으로 분류하는 방법을 배웠습니다. 

입력은 문장(시퀀스)이었지만, 출력은 단 하나의 값(클래스)이었죠. 하지만 만약 우리가 '나는 학생이다'라는 한국어 문장을 'I am a student'라는 영어 문장으로 바꾸고 싶다면 어떻게 해야 할까요? 

이처럼 `하나의 시퀀스를 입력받아, 완전히 다른 새로운 시퀀스를 출력`하는 과업을 `시퀀스-투-시퀀스(Sequence-to-Sequence, Seq2Seq)` 라고 합니다.

Seq2Seq는 기계 번역뿐만 아니라 챗봇(질문 → 답변), 문서 요약(긴 글 → 짧은 글) 등 현대 자연어 처리의 수많은 응용 분야에서 핵심적인 역할을 담당하는 모델 구조입니다. 

RNN/LSTM이 시퀀스 데이터를 '읽는 법'을 배웠다면, Seq2Seq는 시퀀스를 '이해하고 새로운 시퀀스를 쓰는 법'을 배우는 단계라고 할 수 있습니다.

하지만 여기에는 한 가지 큰 난관이 있습니다. 

'나는 학생이다'라는 문장의 모든 정보를 어떻게 하나의 '보따리'에 담아 영어 문장을 생성하는 모델에게 전달할 수 있을까요? 문장이 길어지면 정보가 손실되기 쉽습니다. 

이 문제를 해결하기 위해 등장한 것이 바로 '`어텐션(Attention)`' 메커니즘입니다. 

어텐션은 번역할 단어를 생성하는 매 순간, 원본 문장의 가장 관련 있는 부분에 '집중(Attention)'하여 정보를 가져오는 혁신적인 아이디어입니다.

이번 파트에서는 Seq2Seq의 기본 구조인 `인코더-디코더`부터, 그 한계를 극복하고 성능을 비약적으로 향상시킨 `어텐션 메커니즘`까지 깊이 있게 다룹니다. 

최종적으로는 이 모든 것을 PyTorch 코드로 직접 구현하여, 입력된 숫자의 순서를 뒤집는 간단한 '문장 뒤집기' 예제를 통해 Seq2Seq 모델의 작동 원리를 완벽하게 이해하는 것을 목표로 합니다. 

이 경험은 Part 3에서 배울 트랜스포머의 심장부를 이해하는 데 튼튼한 발판이 될 것입니다.

`이번 파트의 학습 목표:`

* Seq2Seq 모델의 필요성과 기계 번역, 챗봇 등 주요 응용 분야를 이해합니다.

* `인코더(Encoder)-디코더(Decoder)` 구조의 역할과 정보가 `컨텍스트 벡터(Context Vector)`를 통해 전달되는 과정을 설명할 수 있습니다.
* 컨텍스트 벡터의 한계점(정보 압축 손실)을 설명하고, 이를 해결하기 위한 `어텐션(Attention)`의 필요성을 이해합니다.
* `루옹 어텐션(Luong Attention)`의 작동 원리를 이해하고, 쿼리, 키, 밸류의 개념을 바탕으로 어텐션 스코어와 컨텍스트 벡터가 계산되는 과정을 설명할 수 있습니다.
* PyTorch를 사용하여 Bi-LSTM 기반의 인코더, 어텐션, 그리고 어텐션이 적용된 디코더를 각각 구현할 수 있습니다.
* `교사 강요(Teacher Forcing)`의 개념과 역할을 이해하고, Seq2Seq 모델의 학습 과정에 어떻게 사용되는지 설명할 수 있습니다.
* 전체 Seq2Seq 모델을 조립하고, '문장 뒤집기' 예제를 통해 학습 및 추론(Inference) 과정을 처음부터 끝까지 수행할 수 있습니다.

### 1. Seq2Seq의 기본 구조: 인코더-디코더

Seq2Seq 모델은 크게 두 부분, `인코더`와 `디코더`로 구성됩니다. 마치 외국어 번역가처럼 일하죠.

* `인코더 (Encoder):` 번역가는 먼저 한국어 문장 전체를 읽고 그 의미를 완벽하게 파악합니다. 인코더는 이 역할처럼 입력 시퀀스(예: "나는 학생이다")를 하나씩 읽어들여, 문장의 모든 정보를 압축한 하나의 벡터, 즉 `컨텍스트 벡터(Context Vector)` 또는 '생각 벡터(Thought Vector)'를 만듭니다.

* `디코더 (Decoder):` 번역가는 파악한 의미를 바탕으로 영어 문장을 한 단어씩 써 내려갑니다. 디코더는 인코더가 전달한 컨텍스트 벡터를 받아서, 출력 시퀀스(예: "I am a student")를 한 타임스텝에 한 단어씩 생성합니다.

    <img src="https://wikidocs.net/images/page/24996/%EC%9D%B8%EC%BD%94%EB%8D%94%EB%94%94%EC%BD%94%EB%8D%94%EB%AA%A8%EB%8D%B8.PNG" width="700">

인코더의 마지막 은닉 상태가 바로 컨텍스트 벡터가 되어 디코더의 첫 번째 은닉 상태로 전달됩니다. 디코더는 이 컨텍스트 벡터와 문장의 시작을 알리는 `<SOS>`(Start of Sentence) 토큰을 입력받아 첫 번째 단어 "I"를 예측하고, 그 다음엔 "I"를 입력으로 받아 "am"을 예측하는 과정을 반복합니다.

### 2. 한계와 돌파구: 어텐션 메커니즘

이 단순한 인코더-디코더 구조는 명확한 한계를 가집니다. 바로 `컨텍스트 벡터의 정보 병목 현상`입니다. 

아무리 긴 문장이라도 모든 정보를 고정된 크기의 벡터 하나에 욱여넣어야 합니다. 

문장이 길어질수록, 초반부의 중요한 정보가 손실될 가능성이 커집니다. 100페이지짜리 책을 한 문단으로 요약하라고 하는 것과 마찬가지죠.

`어텐션(Attention)`은 이 문제를 해결하기 위해 등장했습니다. 

어텐션의 핵심 아이디어는 `"디코더가 단어를 생성할 때마다, 원본 문장 전체를 다시 한번 훑어보고 지금 생성할 단어와 가장 관련이 깊은 부분에 '집중'하자"`는 것입니다.

<img src="https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdna%2Fb34r3s%2Fbtrj7R5LRBq%2FAAAAAAAAAAAAAAAAAAAAAIImw5xCjvBVdSDZXtGlmfrRKbQoHl4tK41pkZTier0a%2Fimg.png%3Fcredential%3DyqXZFxpELC7KVnFOS48ylbz2pIh7yKj8%26expires%3D1751295599%26allow_ip%3D%26allow_referer%3D%26signature%3DlV4axTE8q8uqhBCkES0wrO9QqUU%253D" width="600">

디코더는 매 타임스텝마다 자신의 현재 은닉 상태(Query)를 가지고, 인코더의 모든 타임스텝별 출력(Key-Value 쌍)을 살펴봅니다. 

그리고 각 인코더 출력이 현재 디코더의 예측에 얼마나 중요한지를 나타내는 '어텐션 스코어'를 계산합니다. 

이 스코어를 가중치로 사용하여 인코더 출력들의 가중합을 구하면, 바로 지금 이 순간에 가장 필요한 맞춤형 컨텍스트 벡터, 즉 `어텐션 컨텍스트 벡터`가 만들어집니다. 

이 방식을 통해 모델은 더 이상 고정된 크기의 컨텍스트 벡터 하나에만 의존하지 않고, 필요할 때마다 원본 시퀀스의 원하는 부분에서 직접 정보를 가져올 수 있게 되어 번역 품질이 비약적으로 향상되었습니다.

### 3. PyTorch로 Seq2Seq 모델 구현하기

이제 이론을 바탕으로 실제 코드를 작성해봅시다. 

#### 3.1. 기본 환경 설정

In [1]:
import math, random, time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import plotly.express as px
import numpy as np

# 장치 설정
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("mps")

# 재현성을 위한 시드 고정
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Using device: {device}")

Using device: mps


#### 3.2. 인코더 (Encoder)
인코더는 입력 시퀀스를 받아 컨텍스트를 압축하는 역할을 합니다. 여기서는 양방향 LSTM (`bidirectional=True`)을 사용하여 문장의 앞에서부터 읽는 것(정방향)과 뒤에서부터 읽는 것(역방향)의 정보를 모두 활용합니다. 이를 통해 각 단어 주변의 문맥을 더 풍부하게 파악할 수 있습니다.

In [2]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers=1, dropout=0.1):
        super().__init__()
        # 단어를 임베딩 벡터로 변환 (padding_idx=0은 패딩 토큰을 0벡터로 만듦)
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        # 양방향 LSTM 레이어
        self.rnn = nn.LSTM(emb_dim, hid_dim, 
                           num_layers=n_layers, dropout=dropout, 
                           bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        # 양방향 LSTM의 은닉 상태(정방향/역방향)를 합친 후, 디코더의 hid_dim에 맞춰주기 위한 FC 레이어
        self.fc = nn.Linear(hid_dim * 2, hid_dim) 

    def forward(self, src, src_len):
        # src: (Batch, Seq_len)
        
        # 1. 임베딩 & 드롭아웃
        embedded = self.dropout(self.embedding(src)) # (B, L, E)
        
        # 2. 패딩 처리를 위한 패킹(Packing)
        # LSTM이 불필요한 <pad> 토큰까지 계산하지 않도록, 실제 길이 정보(src_len)를 바탕으로 시퀀스를 압축
        packed = nn.utils.rnn.pack_padded_sequence(embedded, src_len, 
                                                  batch_first=True, enforce_sorted=False)
        
        # 3. LSTM 통과
        # outputs: 모든 타임스텝의 은닉 상태 (압축된 상태)
        # hidden, cell: 마지막 타임스텝의 (정방향/역방향) 은닉 상태와 셀 상태
        packed_outputs, (h, c) = self.rnn(packed)
        
        # 4. 패킹 해제(Unpacking)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True)
        # outputs: (B, L, 2 * hid_dim) - 모든 스텝의 정/역방향 은닉 상태
        # h, c: (2 * n_layers, B, hid_dim) - 마지막 스텝의 정/역방향 (은닉,셀) 상태
        
        # 5. 디코더의 초기 상태 생성
        # 마지막 은닉/셀 상태의 정방향(h[-2], c[-2])과 역방향(h[-1], c[-1])을 concat하여 FC 레이어 통과
        # 이를 통해 디코더의 초기 은닉 상태와 셀 상태를 생성 (방향성을 통합한 문맥 정보)
        h_cat = torch.cat((h[-2], h[-1]), dim=1)
        c_cat = torch.cat((c[-2], c[-1]), dim=1)
        hidden = torch.tanh(self.fc(h_cat)).unsqueeze(0)
        cell = torch.tanh(self.fc(c_cat)).unsqueeze(0)
        
        return outputs, hidden, cell

#### 3.3. 루옹 어텐션 (Luong-style Attention)

여러 어텐션 방식 중, 비교적 간단하고 효과적인 루옹 어텐션(concat scoring 방식)을 구현합니다. 디코더의 현재 은닉 상태와 인코더의 모든 시점의 출력을 concat한 후, 선형 변환과 tanh 활성화 함수를 거쳐 에너지(스코어)를 계산합니다.

In [3]:
class LuongAttention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        # 디코더 은닉 상태와 인코더 출력 상태를 합친 크기(hid_dim * 3)를 입력받아 스코어를 계산하기 위한 레이어
        # 논문에서는 hid_dim*2 -> hid_dim 이지만, Bi-LSTM 인코더 출력이 hid_dim*2 이므로 총 hid_dim*3
        self.attn = nn.Linear(hid_dim * 3, hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias = False)

    def forward(self, hidden, encoder_outputs, mask):
        # hidden: (B, hid_dim) - 디코더의 현재 은닉 상태
        # encoder_outputs: (B, L_src, hid_dim * 2) - 인코더의 모든 시점 출력
        # mask: (B, L_src) - 인코더 입력의 패딩 마스크
        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]
        
        # 1. 디코더의 현재 은닉 상태를 인코더 시퀀스 길이만큼 복제
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1) # (B, L_src, hid_dim)
        
        # 2. 어텐션 에너지(스코어) 계산
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2))) # (B, L_src, hid_dim)
        
        # 3. 어텐션 스코어를 어텐션 가중치로 변환
        attention = self.v(energy).squeeze(2) # (B, L_src)
        
        # 4. 패딩 토큰에는 어텐션을 주지 않도록 마스킹
        attention = attention.masked_fill(mask == 0, -1e10)
        
        # 5. 소프트맥스를 적용하여 최종 어텐션 가중치 계산
        attn_weights = torch.softmax(attention, dim=1) # (B, L_src)
        
        # 6. 어텐션 가중치와 인코더 출력을 가중합하여 컨텍스트 벡터 생성
        # (B, 1, L_src) @ (B, L_src, hid_dim*2) -> (B, 1, hid_dim*2)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs) 
        
        return context, attn_weights

#### 3.4. 디코더 (Decoder)
디코더는 매 시점마다 이전 타임스텝에서 생성한 단어와 어텐션을 통해 계산된 컨텍스트 벡터를 함께 입력받아 다음 단어를 예측합니다. 즉, LSTM의 입력으로 (이전 단어 임베딩 + 컨텍스트 벡터)가 함께 들어갑니다.

In [4]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, attn, n_layers=1, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        # 어텐션 컨텍스트 벡터(hid_dim*2)와 임베딩(emb_dim)을 합쳐서 입력으로 받음
        self.rnn = nn.LSTM(emb_dim + hid_dim * 2, hid_dim,
                           num_layers=n_layers, dropout=dropout,
                           batch_first=True)
        # 최종 단어 예측을 위한 FC 레이어. LSTM 출력, 컨텍스트 벡터, 현재 임베딩을 모두 사용
        self.fc_out = nn.Linear(hid_dim * 3 + emb_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.attn = attn

    def forward(self, input_tok, hidden, cell, encoder_outputs, mask):
        # input_tok: (B) - 이전 스텝에서 예측된 단어
        # hidden, cell: 디코더의 이전 스텝 (은닉, 셀) 상태
        # encoder_outputs: 인코더의 모든 시점 출력
        # mask: 인코더 입력의 패딩 마스크
        
        input_tok = input_tok.unsqueeze(1) # (B, 1)
        embedded = self.dropout(self.embedding(input_tok)) # (B, 1, E)
        
        # 1. 어텐션 계산: 디코더의 이전 은닉 상태와 인코더 출력을 사용
        context, attn_weights = self.attn(hidden.squeeze(0), encoder_outputs, mask) # context: (B, 1, hid*2)
        
        # 2. LSTM 입력 준비: 어텐션 컨텍스트와 임베딩 벡터를 concat
        rnn_input = torch.cat((embedded, context), dim=2) # (B, 1, E + hid*2)
        
        # 3. LSTM 통과
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        
        # 4. 최종 예측: 필요한 모든 벡터(LSTM 출력, 컨텍스트, 임베딩)를 concat하여 FC 레이어 통과
        output = output.squeeze(1)
        context = context.squeeze(1)
        embedded = embedded.squeeze(1)
        pred = self.fc_out(torch.cat((output, context, embedded), dim=1)) # (B, vocab_size)
        
        return pred, hidden, cell, attn_weights

#### 3.5. Seq2Seq 모델 래퍼 (Wrapper)

마지막으로 인코더와 디코더를 하나로 묶어 전체 Seq2Seq 모델을 완성합니다. 이 모델은 학습 과정 전체를 관장합니다.

`교사 강요(Teacher Forcing)`: 학습 초기에는 모델이 잘못된 단어를 예측하면 그 뒤의 예측이 줄줄이 틀리는 문제가 발생할 수 있습니다. 이를 방지하기 위해, 일정 확률로 디코더의 입력에 모델 자신의 예측(오답일 수 있는) 대신 `실제 정답 단어`를 넣어주는 기법을 사용합니다. 이를 교사 강요라고 하며, 학습을 안정화하고 수렴 속도를 높이는 데 도움이 됩니다.

In [5]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, sos_idx, eos_idx, pad_idx):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.sos_idx, self.eos_idx, self.pad_idx = sos_idx, eos_idx, pad_idx

    def create_mask(self, src):
        # PAD 토큰(0)인 위치는 False, 나머지는 True인 마스크 생성
        return (src != self.pad_idx)

    def forward(self, src, src_len, tgt, teacher_forcing_ratio=0.5):
        # src: (B, L_src), src_len: (B), tgt: (B, L_tgt)
        B, max_len_tgt = tgt.shape
        vocab_size = self.decoder.fc_out.out_features
        
        # 디코더의 출력을 저장할 텐서
        outputs = torch.zeros(B, max_len_tgt, vocab_size, device=device)
        
        # 1. 인코더 통과
        enc_out, hidden, cell = self.encoder(src, src_len)
        mask = self.create_mask(src)
        
        # 2. 디코더의 첫 입력으로 <SOS> 토큰 사용
        input_tok = tgt[:, 0]
        
        # 3. 타겟 시퀀스 길이만큼 디코더를 반복 실행
        for t in range(1, max_len_tgt):
            # 디코더를 통해 다음 단어 예측
            pred, hidden, cell, _ = self.decoder(input_tok, hidden, cell, enc_out, mask)
            outputs[:, t] = pred
            
            # 교사 강요 적용 여부 결정
            teacher_force = random.random() < teacher_forcing_ratio
            
            # 가장 확률이 높은 단어를 다음 입력으로 사용
            top1 = pred.argmax(1)
            
            # 교사 강요가 켜져 있으면 실제 정답을, 꺼져 있으면 모델의 예측을 다음 입력으로 사용
            input_tok = tgt[:, t] if teacher_force else top1
        
        return outputs

### 4. Toy Dataset으로 학습하기: 문장 뒤집기

복잡한 번역 데이터 대신, 간단한 '문장 뒤집기' 태스크로 모델의 작동 원리를 확인해 보겠습니다. 예를 들어 `[4, 7, 9, 5]` 라는 시퀀스가 입력되면, `[5, 9, 7, 4]`를 출력하도록 학습시키는 것입니다. 이를 통해 Seq2Seq와 어텐션이 시퀀스의 순서와 관계를 학습하는 과정을 명확히 볼 수 있습니다.

In [6]:
import random, math, time
from typing import Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------- 1. Data ---------------------------- #
PAD, SOS, EOS = 0, 1, 2          # special tokens
VOCAB_SIZE = 50                  # 0~49 (0 is PAD)
MIN_LEN, MAX_LEN = 4, 10

class SyntheticDataset(Dataset):
    """입력 시퀀스를 뒤집은 시퀀스를 타깃으로 반환하는 Toy 데이터셋"""
    def __init__(self, n_samples: int = 10_000):
        self.data = []
        for _ in range(n_samples):
            seq_len = random.randint(MIN_LEN, MAX_LEN)
            tokens = [random.randint(3, VOCAB_SIZE - 1) for _ in range(seq_len)]
            src = [SOS] + tokens + [EOS]
            tgt = [SOS] + tokens[::-1] + [EOS]
            self.data.append((src, tgt))

    def __len__(self): return len(self.data)

    def __getitem__(self, idx): return self.data[idx]

def pad_batch(batch, pad_token=PAD) -> Tuple[torch.Tensor, torch.Tensor]:
    """Collate fn → (src_batch, tgt_batch) [B, L]"""
    srcs, tgts = zip(*batch)
    max_src = max(len(s) for s in srcs)
    max_tgt = max(len(t) for t in tgts)
    src_pad = [s + [pad_token]*(max_src-len(s)) for s in srcs]
    tgt_pad = [t + [pad_token]*(max_tgt-len(t)) for t in tgts]
    return torch.tensor(src_pad, dtype=torch.long), torch.tensor(tgt_pad, dtype=torch.long)

train_loader = DataLoader(SyntheticDataset(9000), batch_size=128, shuffle=True, collate_fn=pad_batch)
valid_loader = DataLoader(SyntheticDataset(1000), batch_size=128, shuffle=False, collate_fn=pad_batch)

# ---------------------------- 2. Model ---------------------------- #
EMB_DIM = 64
HID_DIM = 128
ATTN_DIM = 128
N_LAYERS = 1
DROPOUT = 0.1

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, EMB_DIM, padding_idx=PAD)
        self.rnn = nn.GRU(EMB_DIM, HID_DIM, N_LAYERS, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(HID_DIM*2, HID_DIM)   # combine directions
    def forward(self, src, src_lens):
        embedded = self.embed(src)
        packed = nn.utils.rnn.pack_padded_sequence(embedded, src_lens, batch_first=True, enforce_sorted=False)
        outputs, hidden = self.rnn(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs, batch_first=True)
        # concat forward & backward → pass through fc to get [N, HID]
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2], hidden[-1]), dim=1))).unsqueeze(0)  # [1,B,H]
        return outputs, hidden                        # outputs: [B,L,2H], hidden: [1,B,H]

class BahdanauAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.W_enc = nn.Linear(HID_DIM*2, ATTN_DIM, bias=False)
        self.W_dec = nn.Linear(HID_DIM, ATTN_DIM, bias=False)
        self.v = nn.Linear(ATTN_DIM, 1, bias=False)
    def forward(self, decoder_hidden, encoder_outputs, mask):         # hidden: [B,H]
        # encoder_outputs: [B,L,2H] → score: [B,L]
        score = self.v(torch.tanh(self.W_enc(encoder_outputs) + self.W_dec(decoder_hidden).unsqueeze(1))).squeeze(-1)
        score = score.masked_fill(mask == 0, -1e9)
        attn_weights = torch.softmax(score, dim=1)                    # [B,L]
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)  # [B,2H]
        return context, attn_weights

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, EMB_DIM, padding_idx=PAD)
        self.attn = BahdanauAttention()
        self.rnn = nn.GRU(EMB_DIM + HID_DIM*2, HID_DIM, N_LAYERS, batch_first=True)
        self.fc_out = nn.Linear(HID_DIM*3 + EMB_DIM, VOCAB_SIZE)
        self.dropout = nn.Dropout(DROPOUT)
    def forward(self, input_tok, hidden, encoder_outputs, mask):
        # input_tok: [B], hidden: [1,B,H]
        embedded = self.dropout(self.embed(input_tok)).unsqueeze(1)   # [B,1,E]
        context, _ = self.attn(hidden.squeeze(0), encoder_outputs, mask)  # context: [B,2H]
        rnn_input = torch.cat((embedded, context.unsqueeze(1)), dim=2)    # [B,1,E+2H]
        output, hidden = self.rnn(rnn_input, hidden)                      # output: [B,1,H]
        output = output.squeeze(1)
        pred = self.fc_out(torch.cat((output, context, embedded.squeeze(1)), dim=1))  # [B, V]
        return pred, hidden

class Seq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = Encoder()
        self.dec = Decoder()
    def create_mask(self, src):
        return (src != PAD).to(src.device)
    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size, tgt_len = tgt.shape
        outputs = torch.zeros(tgt_len, batch_size, VOCAB_SIZE, device=src.device)
        src_lens = (src != PAD).sum(1).cpu()
        enc_out, hidden = self.enc(src, src_lens)
        mask = self.create_mask(src)
        input_tok = tgt[:,0]                   # <sos>
        for t in range(1, tgt_len):
            pred, hidden = self.dec(input_tok, hidden, enc_out, mask)
            outputs[t] = pred
            teacher = random.random() < teacher_forcing_ratio
            input_tok = tgt[:,t] if teacher else pred.argmax(1)
        return outputs


### 5. 모델 학습
이제 모든 준비가 끝났습니다. 모델의 하이퍼파라미터를 설정하고, 데이터로더를 생성하여 학습 루프를 실행합니다.

In [7]:

# ---------------------------- 3. Training ---------------------------- #
model = Seq2Seq().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)

def epoch_step(loader, train=True):
    model.train() if train else model.eval()
    epoch_loss = 0
    with torch.set_grad_enabled(train):
        for src, tgt in loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)

            output = model(src, tgt, teacher_forcing_ratio=0.5 if train else 0)
            output_dim = output.shape[-1]

            # <sos> 토큰 제외
            output = output[1:].permute(1, 0, 2).reshape(-1, output_dim)  # [B*(T-1), V]
            tgt_flat = tgt[:, 1:].reshape(-1)                              # [B*(T-1)]

            loss = criterion(output, tgt_flat)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
                optimizer.step()

            epoch_loss += loss.item()
    return epoch_loss / len(loader)



N_EPOCH = 12
for epoch in range(1, N_EPOCH+1):
    train_loss = epoch_step(train_loader, train=True)
    valid_loss = epoch_step(valid_loader, train=False)
    print(f"Epoch {epoch:02}/{N_EPOCH} | Train {train_loss:.3f} | Val {valid_loss:.3f}")


Epoch 01/12 | Train 2.719 | Val 0.772
Epoch 02/12 | Train 0.297 | Val 0.109
Epoch 03/12 | Train 0.096 | Val 0.046
Epoch 04/12 | Train 0.061 | Val 0.620
Epoch 05/12 | Train 0.045 | Val 0.065
Epoch 06/12 | Train 0.033 | Val 0.008
Epoch 07/12 | Train 0.041 | Val 0.007
Epoch 08/12 | Train 0.027 | Val 0.004
Epoch 09/12 | Train 0.020 | Val 0.005
Epoch 10/12 | Train 0.017 | Val 0.004
Epoch 11/12 | Train 0.019 | Val 0.055
Epoch 12/12 | Train 0.009 | Val 0.002


In [8]:

# ---------------------------- 4. Demo ---------------------------- #
def reverse_infer(seq):
    model.eval()
    with torch.no_grad():
        src = torch.tensor([SOS]+seq+[EOS], dtype=torch.long).unsqueeze(0).to(DEVICE)
        tgt_len = len(seq)+2
        outputs = torch.zeros(tgt_len, 1, VOCAB_SIZE, device=DEVICE)
        enc_out, hidden = model.enc(src, torch.tensor([len(src[0])]))
        mask = model.create_mask(src)
        input_tok = torch.tensor([SOS], device=DEVICE)
        preds = []
        for _ in range(1, tgt_len):
            pred, hidden = model.dec(input_tok, hidden, enc_out, mask)
            input_tok = pred.argmax(1)
            if input_tok.item() == EOS: break
            preds.append(input_tok.item())
        return preds

print("--- SAMPLE PREDICTIONS ---")
for _ in range(5):
    seq_len = random.randint(MIN_LEN, MAX_LEN)
    sample = [random.randint(3, VOCAB_SIZE-1) for _ in range(seq_len)]
    pred = reverse_infer(sample)
    print(f"INPUT   : {sample}")
    print(f"EXPECTED: {list(reversed(sample))}")
    print(f"PREDICT : {pred}\n")

--- SAMPLE PREDICTIONS ---
INPUT   : [45, 49, 3, 40, 24, 10, 41, 49, 38, 4]
EXPECTED: [4, 38, 49, 41, 10, 24, 40, 3, 49, 45]
PREDICT : [4, 38, 49, 41, 10, 24, 40, 3, 49, 45]

INPUT   : [23, 41, 42, 6, 25, 27, 4, 16]
EXPECTED: [16, 4, 27, 25, 6, 42, 41, 23]
PREDICT : [16, 4, 27, 25, 6, 42, 41, 23]

INPUT   : [11, 27, 28, 11, 49, 47, 24, 49, 9, 6]
EXPECTED: [6, 9, 49, 24, 47, 49, 11, 28, 27, 11]
PREDICT : [6, 9, 49, 24, 47, 49, 11, 28, 27, 11]

INPUT   : [9, 26, 18, 4, 35]
EXPECTED: [35, 4, 18, 26, 9]
PREDICT : [35, 4, 18, 26, 9]

INPUT   : [5, 18, 17, 3, 13, 20, 29]
EXPECTED: [29, 20, 13, 3, 17, 18, 5]
PREDICT : [29, 20, 13, 3, 17, 18, 5]



### 6. 추론 (Inference) 및 어텐션 시각화

학습된 모델을 사용하여 새로운 입력 시퀀스를 뒤집어보고, 어텐션 가중치를 시각화하여 모델이 어떤 입력에 '집중'하는지 확인해 봅시다.

In [9]:
def greedy_decode(model, src_seq, max_len=20):
    model.eval()
    with torch.no_grad():
        src_tensor = torch.tensor([SOS] + src_seq + [EOS], dtype=torch.long).unsqueeze(0).to(DEVICE)
        src_len_tensor = torch.tensor([len(src_tensor[0])])
        
        enc_out, hidden = model.enc(src_tensor, src_len_tensor)
        mask = model.create_mask(src_tensor)

        input_tok = torch.tensor([SOS], device=DEVICE)
        generated = []
        attention_history = []

        for _ in range(max_len):
            pred, hidden = model.dec(input_tok, hidden, enc_out, mask)
            top1 = pred.argmax(1).item()
            
            if top1 == EOS:
                break
            
            generated.append(top1)
            # 어텐션 가중치 저장 (모델에서 반환하는 경우)
            attention_history.append(pred.cpu().numpy())
            input_tok = torch.tensor([top1], device=DEVICE)
            
        return generated, np.array(attention_history)

# 테스트 예시
ex_input = [4, 7, 9, 5, 12, 15, 3]
print("Input :", ex_input)

output_seq, attention_matrix = greedy_decode(model, ex_input)
print("Output:", output_seq)

# 어텐션 시각화 (간단한 출력 확률 히트맵)
if len(output_seq) > 0:
    import plotly.express as px
    import numpy as np
    
    # 출력 시퀀스의 확률 분포를 시각화
    attention_matrix = attention_matrix[:len(output_seq), :VOCAB_SIZE]
    
    # 3D 배열을 2D로 변환 (첫 번째 차원을 평면화)
    attention_matrix_2d = attention_matrix.reshape(attention_matrix.shape[0], -1)
    
    fig = px.imshow(attention_matrix_2d, 
                    labels=dict(x="Vocabulary", y="Generated Sequence", color="Probability"),
                    title="Output Probability Distribution Heatmap")
    fig.update_xaxes(side="top")
    fig.show()

Input : [4, 7, 9, 5, 12, 15, 3]
Output: [3, 15, 12, 5, 9, 7, 4]


어텐션 히트맵을 보면, 디코더가 출력 시퀀스의 각 토큰을 생성할 때마다 전체 어휘(Vocabulary)에 대한 확률 분포가 어떻게 변화하는지 시각적으로 확인할 수 있습니다.

위 히트맵에서 y축은 생성된 시퀀스의 각 토큰(Generated Sequence), x축은 전체 어휘(Vocabulary)를 의미합니다. 색이 밝을수록 해당 어휘에 높은 확률(또는 어텐션 가중치)이 할당된 것을 나타냅니다.

예를 들어, 첫 번째 행(0번 행)은 첫 번째 출력 토큰을 생성할 때의 확률 분포를 보여주며, 특정 어휘(예: 3번 토큰)에서 가장 밝은 색을 띄는 것을 볼 수 있습니다. 이는 디코더가 첫 번째 토큰을 생성할 때 입력 시퀀스의 마지막 단어(3)에 가장 집중했다는 의미입니다.

이처럼 각 행을 따라가며 보면, 디코더가 출력 토큰을 하나씩 생성할 때마다 입력 시퀀스의 어떤 부분(어휘)에 주목하는지, 그리고 '뒤집기'라는 과업을 성공적으로 학습했는지 직관적으로 파악할 수 있습니다.

즉, 어텐션 메커니즘이 디코더가 필요한 정보에 동적으로 집중하도록 도와주며, 실제로 입력 시퀀스를 거꾸로 복원하는 데 핵심적인 역할을 하고 있음을 이 히트맵을 통해 명확히 알 수 있습니다.



### 📝 마무리

이번 파트에서는 시퀀스를 다른 시퀀스로 변환하는 Seq2Seq 모델과, 그 핵심 동력인 어텐션 메커니즘을 바닥부터 구현해보았습니다. 

* `인코더-디코더 구조`는 입력의 의미를 압축하고, 이를 바탕으로 출력을 생성하는 기본적인 프레임워크를 제공합니다.
* `어텐션`은 정보 병목 현상을 해결하고, 디코더가 동적으로 필요한 정보에 집중할 수 있게 하여 모델의 성능을 크게 향상시켰습니다.

이러한 Seq2Seq와 어텐션의 개념은 여기서 멈추지 않습니다. 

다음 파트에서 배울 `트랜스포머(Transformer)`는 RNN의 순차적인 구조를 완전히 제거하고 오직 이 '어텐션' 메커니즘만을 사용하여 한 단계 더 발전된 성능과 효율성을 달성합니다. 

오늘 배운 어텐션의 원리가 바로 트랜스포머를 이해하는 가장 중요한 열쇠가 될 것입니다.